# Rerun of SFU author collection

* Originally were getting 20k+ authors when using author search in OpenAlex
* This script was provided to us by the OpenAlex team
* We are interested in all authors who published with SFU between 2020-2024
* As a sanity check, Scopus lists ~6k SFU authors in this time frame, so we would expect somewhat more authors to be returned here (but <<20k)

In [35]:
import urllib.request, json, urllib.parse 
import requests
import pandas as pd
import ast
from tqdm import tqdm

In [36]:
SFU = "https://openalex.org/I18014758" 
#base = ("https://api.openalex.org/works?" "filter=authorships.institutions.id:I18014758,publication_year:2020-2024" "&select=authorships&per_page=200&mailto=tpa42@sfu.ca") 
base = ("https://api.openalex.org/works?" "filter=authorships.institutions.id:I18014758,publication_year:2020-2024" "&per_page=200&mailto=tpa42@sfu.ca") 
cursor, authors, works = "*", set(), []

while cursor: 
    d = json.load(urllib.request.urlopen(base + "&cursor=" + urllib.parse.quote(cursor))) 
    if not d["results"]: 
        break 
    for w in d["results"]:
        works.append(w)
        for au in w["authorships"]: 
            if SFU in {i["id"] for i in au["institutions"]}: 
                authors.add(au["author"]["id"]) 
    cursor = d["meta"]["next_cursor"] 

In [37]:
print(w)

{'id': 'https://openalex.org/W7165396956', 'doi': 'https://doi.org/10.70177/jsa.v1i6.1674', 'title': 'The Impact of Selective Logging on Forest Structure and Function', 'display_name': 'The Impact of Selective Logging on Forest Structure and Function', 'publication_year': 2024, 'publication_date': '2024-12-26', 'ids': {'openalex': 'https://openalex.org/W7165396956', 'doi': 'https://doi.org/10.70177/jsa.v1i6.1674'}, 'language': None, 'primary_location': {'id': 'doi:10.70177/jsa.v1i6.1674', 'is_oa': True, 'landing_page_url': 'https://doi.org/10.70177/jsa.v1i6.1674', 'pdf_url': 'https://research.adra.ac.id/index.php/selvicoltura/article/download/1674/1128', 'source': {'id': 'https://openalex.org/S7407059167', 'display_name': 'Journal of Selvicoltura Asean', 'issn_l': '3048-1171', 'issn': ['3048-1171', '3048-1198'], 'is_oa': False, 'is_in_doaj': False, 'is_core': False, 'host_organization': None, 'host_organization_name': None, 'host_organization_lineage': [], 'host_organization_lineage_na

we check the length of the new author list, and it is in line with what we would expect to see :)

In [38]:
print(len(authors)) # -> ~7,900 
print(len(works))

7929
18779


In [39]:
list(authors).count(None)

1

looking at the list - we get a bunch of openalex author ids. now need to loop through this entire list, query the author API, then collect useful information about each author from there

In [40]:
print(authors, 10)

{'https://openalex.org/A5102170192', 'https://openalex.org/A5038053101', 'https://openalex.org/A5012451717', 'https://openalex.org/A5023286690', 'https://openalex.org/A5030365370', 'https://openalex.org/A5058193230', 'https://openalex.org/A5113834078', 'https://openalex.org/A5051528851', 'https://openalex.org/A5034377021', 'https://openalex.org/A5052364420', 'https://openalex.org/A5031424991', 'https://openalex.org/A5110631057', 'https://openalex.org/A5032810944', 'https://openalex.org/A5107455662', 'https://openalex.org/A5005480436', 'https://openalex.org/A5088872913', 'https://openalex.org/A5021515928', 'https://openalex.org/A5110711132', 'https://openalex.org/A5037654364', 'https://openalex.org/A5055327388', 'https://openalex.org/A5087695450', 'https://openalex.org/A5114395731', 'https://openalex.org/A5091636310', 'https://openalex.org/A5040749724', 'https://openalex.org/A5037912444', 'https://openalex.org/A5052216983', 'https://openalex.org/A5059801247', 'https://openalex.org/A5022

In [41]:
authors_detailed = [] 

for au in tqdm(authors):
    if au != None: 
        id = str(au).lstrip("https://openalex.org/")

        url = f"https://api.openalex.org/authors/{id}"
        results = requests.get(url)

        try:
            auth = results.json()

        except:
            msg = f"Error with ID: {id}"
            print(msg)

        else: 
            try:
                authors_detailed.append(
                    {'OpenAlex ID': id, 
                    'ORCID': auth['orcid'], 
                    'Name': auth['display_name'], 
                    'Alt Names': auth['raw_author_names'], 
                    'Publications': auth['works_count'], 
                    'Citations': auth['cited_by_count'], 
                    'h-index': auth['summary_stats']['h_index'], 
                    'Number of Institutions': len(auth['affiliations']),
                    'Topic': auth['topic_share'][0]['domain']['display_name']}
                )
            except: 
                authors_detailed.append(
                    {'OpenAlex ID': id, 
                    'ORCID': auth['orcid'], 
                    'Name': auth['display_name'], 
                    'Alt Names': auth['raw_author_names'], 
                    'Publications': auth['works_count'], 
                    'Citations': auth['cited_by_count'], 
                    'h-index': auth['summary_stats']['h_index'], 
                    'Number of Institutions': len(auth['affiliations']),
                    'Topic': ''}
                )

authors_detailed = pd.DataFrame(authors_detailed)
authors_detailed


100%|██████████| 7929/7929 [30:37<00:00,  4.32it/s]  


,OpenAlex ID,ORCID,Name,Alt Names,Publications,Citations,h-index,Number of Institutions,Topic
0,A5102170192,None,Jeongmee Kim,[Jeongmee Kim],4,10,2,2,Health Sciences
1,A5038053101,None,Nicole Szulc,[Nicole Szulc],1,38,1,2,Life Sciences
2,A5012451717,https://orcid.org/0000-0002-2613-9321,M. Williams,"[M C Williams, M Williams, M. C. S. Williams, ...",73,1696,18,16,Physical Sciences
3,A5023286690,https://orcid.org/0000-0002-1107-9135,Emma Griffiths,"[E Griffiths, E. Griffiths, E.J. Griffiths, Em...",119,9726,31,21,Life Sciences
4,A5030365370,https://orcid.org/0000-0002-9012-4079,Alfons A den Broeder,"[A A Den Broeder, A A d. Broeder, A A den Broe...",353,11851,48,31,Health Sciences
...,...,...,...,...,...,...,...,...,...
7923,A5108220854,None,D. H. C. Wilton,"[D H C Wilton, D Wilton, D. H. C. Wilton, D. H...",65,639,14,8,Physical Sciences
7924,A5022991962,https://orcid.org/0000-0001-7496-3991,Jessica Bouchard,"[Bouchard, Jessica, J. R. Bouchard, J.R. Bouch...",40,400,12,6,Social Sciences
7925,A5000439771,None,Priya Nahal,[Priya Nahal],1,11,1,1,Social Sciences
7926,A5032945153,https://orcid.org/0000-0001-8866-6541,Kirsten Zickfeld,"[K Zickfeld, K. Zickfeld, Kirsten Zickfeld, Zi...",195,8744,47,5,Physical Sciences


In [42]:
auth['topic_share'][0]['domain']['display_name']

'Physical Sciences'

In [43]:
test = pd.DataFrame(works)
test

,id,doi,title,display_name,publication_year,publication_date,ids,language,primary_location,type,...,funders,has_content,content_urls,referenced_works_count,referenced_works,related_works,abstract_inverted_index,counts_by_year,updated_date,created_date
0,https://openalex.org/W4288079944,https://doi.org/10.1051/0004-6361/201833910,<i>Planck</i> 2018 results,<i>Planck</i> 2018 results,2020,2020-04-03,{'openalex': 'https://openalex.org/W4288079944...,en,"{'id': 'doi:10.1051/0004-6361/201833910', 'is_...",article,...,"[{'id': 'https://openalex.org/F4320306101', 'd...","{'pdf': True, 'grobid_xml': True}",{'pdf': 'https://content.openalex.org/works/W4...,528,"[https://openalex.org/W1481932901, https://ope...","[https://openalex.org/W2047028841, https://ope...","{'We': [0, 90, 264, 432], 'present': [1], 'cos...","[{'year': 2026, 'cited_by_count': 1149}, {'yea...",2026-06-29T08:53:18.405633,2022-07-28T00:00:00
1,https://openalex.org/W3004480399,https://doi.org/10.1038/s41586-020-1943-3,The repertoire of mutational signatures in hum...,The repertoire of mutational signatures in hum...,2020,2020-02-05,{'openalex': 'https://openalex.org/W3004480399...,en,"{'id': 'doi:10.1038/s41586-020-1943-3', 'is_oa...",article,...,"[{'id': 'https://openalex.org/F4320308349', 'd...","{'pdf': True, 'grobid_xml': True}",{'pdf': 'https://content.openalex.org/works/W3...,66,"[https://openalex.org/W1246381107, https://ope...","[https://openalex.org/W4255048859, https://ope...","{'Abstract': [0], 'Somatic': [1], 'mutations':...","[{'year': 2026, 'cited_by_count': 236}, {'year...",2026-06-29T08:53:18.405633,2025-10-10T00:00:00
2,https://openalex.org/W3006500278,https://doi.org/10.1038/s41586-020-1969-6,Pan-cancer analysis of whole genomes,Pan-cancer analysis of whole genomes,2020,2020-02-05,{'openalex': 'https://openalex.org/W3006500278...,en,"{'id': 'doi:10.1038/s41586-020-1969-6', 'is_oa...",article,...,"[{'id': 'https://openalex.org/F4320311543', 'd...","{'pdf': True, 'grobid_xml': True}",{'pdf': 'https://content.openalex.org/works/W3...,113,"[https://openalex.org/W1517051009, https://ope...","[https://openalex.org/W4391375266, https://ope...","{'Abstract': [0], 'Cancer': [1, 58, 64], 'is':...","[{'year': 2026, 'cited_by_count': 166}, {'year...",2026-06-29T08:53:18.405633,2020-02-24T00:00:00
3,https://openalex.org/W3108936441,https://doi.org/10.1080/15548627.2020.1797280,Guidelines for the use and interpretation of a...,Guidelines for the use and interpretation of a...,2021,2021-01-02,{'openalex': 'https://openalex.org/W3108936441...,en,"{'id': 'doi:10.1080/15548627.2020.1797280', 'i...",article,...,"[{'id': 'https://openalex.org/F4320306127', 'd...","{'pdf': True, 'grobid_xml': False}",{'pdf': 'https://content.openalex.org/works/W3...,4092,"[https://openalex.org/W1919353, https://openal...","[https://openalex.org/W4391375266, https://ope...","{'autophagic': [0], 'responses.': [1], 'Here,'...","[{'year': 2026, 'cited_by_count': 213}, {'year...",2026-06-29T08:53:18.405633,2020-12-07T00:00:00
4,https://openalex.org/W3045910546,https://doi.org/10.1038/s41586-020-2493-4,Expanded encyclopaedias of DNA elements in the...,Expanded encyclopaedias of DNA elements in the...,2020,2020-07-29,{'openalex': 'https://openalex.org/W3045910546...,en,"{'id': 'doi:10.1038/s41586-020-2493-4', 'is_oa...",article,...,"[{'id': 'https://openalex.org/F4320332161', 'd...","{'pdf': True, 'grobid_xml': True}",{'pdf': 'https://content.openalex.org/works/W3...,73,"[https://openalex.org/W1544691147, https://ope...","[https://openalex.org/W3160486573, https://ope...","{'Abstract': [0], 'The': [1], 'human': [2, 115...","[{'year': 2026, 'cited_by_count': 209}, {'year...",2026-06-28T08:01:55.173337,2020-08-03T00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18774,https://openalex.org/W7164925562,https://doi.org/10.1080/17153379.2024.12558497,The Industrial Ephemeral: Labor and Love in In...,The Industrial Ephemeral: Labor and Love in In...,2024,

In [44]:
pd.DataFrame(authors_detailed).to_csv('./OA_auths_20-24.csv', index = False)
pd.DataFrame(works).to_csv('./OA_works_20-24.csv', index = False)

In [45]:
works = pd.DataFrame(works)
sum(works['cited_by_count'])

316165

In [47]:
print(len(works))

18779
